In [ ]:
from langgraph.graph import StateGraph,START,END
from typing import TypedDict, Annotated
from langchain_core.messages import BaseMessage,HumanMessage
from langchain_groq import ChatGroq
from langchain_google_genai import ChatGoogleGenerativeAI

In [ ]:
from dotenv import load_dotenv
load_dotenv()

In [ ]:
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0,
)

In [ ]:
#Desfining state

from langgraph.graph.message import add_messages

class ChatState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]

In [ ]:
# defining nodes
def chat_node(state: ChatState):
    # take user query from the state
    messages = state["messages"]
    
    ## send to llm
    response = llm.invoke(messages)
    #response store state
    return {"messages":[response]}
    

In [ ]:
graph = StateGraph(ChatState)

#Adding nodes

graph.add_node('chat_node',chat_node)

## Add edges
graph.add_edge(START, 'chat_node')
graph.add_edge('chat_node', END)

#Compile the graph
chatbot = graph.compile()





In [ ]:
chatbot

In [ ]:
initial_state ={"messages":  [HumanMessage(content="Tell me about tarakmehta ka ulta chasma ?")] }

response = chatbot.invoke(initial_state)

In [ ]:
for m in response["messages"]:
    m.pretty_print()

In [ ]:
while True:
    user_message = input("Type Here: ").strip()

    if not user_message:
        print("Please enter a message.")
        continue

    if user_message.lower() in ["exit", "quit", "bye"]:
        break

    response = chatbot.invoke({
        "messages": [HumanMessage(content=user_message)]
    })

    print("AI:", response["messages"][-1].content)